# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [1]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests

# Import API key
from api_keys import geoapify_key

In [2]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,keflavik,64.0049,-22.5624,1.24,93,75,5.14,IS,1739062856
1,1,bethel,41.3712,-73.4140,-0.55,62,100,1.79,US,1739062801
2,2,margaret river,-33.9500,115.0667,21.69,69,18,6.32,AU,1739062859
3,3,new plymouth,-39.0667,174.0833,21.84,100,0,4.03,NZ,1739062860
4,4,ushuaia,-54.8000,-68.3000,8.81,71,40,6.69,AR,1739062628


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [19]:
%%capture --no-display

# Configure the map plot
city_data_df.hvplot.points(
    x='Lng',
    y='Lat',
    geo=True,
    c='Humidity',
    colorbar=True,
    title='City Latitude vs. Longitude',
    xlabel='Longitude',
    ylabel='Latitude',
    grid=True,
    width=800,
    height=500
)

# Display the map


:Points   [Lng,Lat]   (Humidity)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [21]:
# Narrow down cities that fit criteria and drop any results with null values
ideal_cities = city_data_df[
    (city_data_df['Max Temp'] < 27) &
    (city_data_df['Max Temp'] > 21) &
    (city_data_df['Wind Speed'] < 4.5) &
    (city_data_df['Cloudiness'] == 0)
]

# Drop any rows with null values
ideal_cities = ideal_cities.dropna()
# Display sample data
ideal_cities


,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
3,3,new plymouth,-39.0667,174.0833,21.84,100,0,4.03,NZ,1739062860
30,30,arica,-18.4750,-70.3042,24.13,73,0,2.06,CL,1739062757
125,125,wailua homesteads,22.0669,-159.3780,25.07,73,0,2.57,US,1739063017
157,157,ciudad lazaro cardenas,17.9583,-102.2000,25.58,74,0,2.37,MX,1739063055
172,172,homestead,25.4687,-80.4776,23.34,83,0,2.06,US,1739063075
193,193,kapa'a,22.0752,-159.3190,25.82,73,0,2.57,US,1739063104
312,312,motueka,-41.1333,173.0167,25.05,46,0,3.67,NZ,1739063252
330,330,lazaro cardenas,17.9583,-102.2000,25.58,74,0,2.37,MX,1739063277
335,335,mandalay,21.9747,96.0836,21.69,49,0,1.22,MM,1739062991
376,376,savanna-la-mar,18.2190,-78.1332,26.07,81,0,0.29,JM,1739063330


### Step 3: Create a new DataFrame called `hotel_df`.

In [22]:
# Use the Pandas copy function to create DataFrame called hotel_df to store the city, country, coordinates, and humidity
hotel_df = ideal_cities[['City', 'Country', 'Lat', 'Lng', 'Humidity']].copy()

# Add an empty column, "Hotel Name," to the DataFrame so you can store the hotel found using the Geoapify API
hotel_df['Hotel Name'] = ''

# Display sample data
hotel_df

,City,Country,Lat,Lng,Humidity,Hotel Name
3,new plymouth,NZ,-39.0667,174.0833,100,
30,arica,CL,-18.4750,-70.3042,73,
125,wailua homesteads,US,22.0669,-159.3780,73,
157,ciudad lazaro cardenas,MX,17.9583,-102.2000,74,
172,homestead,US,25.4687,-80.4776,83,
193,kapa'a,US,22.0752,-159.3190,73,
312,motueka,NZ,-41.1333,173.0167,46,
330,lazaro cardenas,MX,17.9583,-102.2000,74,
335,mandalay,MM,21.9747,96.0836,49,
376,savanna-la-mar,JM,18.2190,-78.1332,81,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [23]:
# Set parameters to search for a hotel
radius = 10000
params = {
    "radius": radius,
    "type": "lodging",
    "key": geoapify_key
}

# Print a message to follow up the hotel search
print("Starting hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    lat = row["Lat"]
    lng = row["Lng"]
    
    # Add the current city's latitude and longitude to the params dictionary
    params["filter"] = f"circle:{radius}@{lat},{lng}"
    params["bias"] = f"proximity:{lat},{lng}"

    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"

    # Make and API request using the params dictionary
    name_address = requests.get(base_url, params=params)

    # Convert the API response to JSON format
    name_address = name_address.json()

    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"

    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df

Starting hotel search
new plymouth - nearest hotel: No hotel found
arica - nearest hotel: No hotel found
wailua homesteads - nearest hotel: No hotel found
ciudad lazaro cardenas - nearest hotel: No hotel found
homestead - nearest hotel: No hotel found
kapa'a - nearest hotel: No hotel found
motueka - nearest hotel: No hotel found
lazaro cardenas - nearest hotel: No hotel found
mandalay - nearest hotel: No hotel found
savanna-la-mar - nearest hotel: No hotel found
cabo san lucas - nearest hotel: No hotel found
pearsall - nearest hotel: No hotel found
amli - nearest hotel: No hotel found
veraval - nearest hotel: No hotel found


,City,Country,Lat,Lng,Humidity,Hotel Name
3,new plymouth,NZ,-39.0667,174.0833,100,No hotel found
30,arica,CL,-18.4750,-70.3042,73,No hotel found
125,wailua homesteads,US,22.0669,-159.3780,73,No hotel found
157,ciudad lazaro cardenas,MX,17.9583,-102.2000,74,No hotel found
172,homestead,US,25.4687,-80.4776,83,No hotel found
193,kapa'a,US,22.0752,-159.3190,73,No hotel found
312,motueka,NZ,-41.1333,173.0167,46,No hotel found
330,lazaro cardenas,MX,17.9583,-102.2000,74,No hotel found
335,mandalay,MM,21.9747,96.0836,49,No hotel found
376,savanna-la-mar,JM,18.2190,-78.1332,81,No hotel found


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [26]:
%%capture --no-display

# Configure the map plot
hotel_map = hotel_df.hvplot.points(
    x='Lng',
    y='Lat',
    geo=True,
    color='blue',
    size=100,
    title='Ideal Cities with Hotels',
    xlabel='Longitude',
    ylabel='Latitude',
    grid=True,
    width=800,
    height=500
)

# Display the map
hotel_map

:Points   [Lng,Lat]